# Notebook 4: SHAP + Counterfactual Analysis
**xAI core — global SHAP, local waterfall plots, DiCE-ML counterfactuals**

Depends on: NB2 + NB3 must both be complete  
Libraries: `shap`, `dice-ml`, `scikit-learn`, `matplotlib`, `seaborn`

## 1. Imports & Setup

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import dice_ml
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
%matplotlib inline
shap.initjs()

## 2. Paths — Edit `DATA_DIR` and `OUTPUT_DIR` to match your project

In [ ]:
# ── Edit these two paths to match your local setup ────────────────────────────
DATA_DIR   = r"C:\Users\uncle\OneDrive\Desktop\comp 6940 proj\COMP6940-Project\data\merged_data"
OUTPUT_DIR = r"C:\Users\uncle\OneDrive\Desktop\comp 6940 proj\COMP6940-Project\artifacts"
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data dir   : {DATA_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

## 3. Load Data

In [ ]:
train = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
test  = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))
val   = pd.read_parquet(os.path.join(DATA_DIR, "val.parquet"))

FEATURE_COLS = [c for c in train.columns if c not in ("salary", "country_name")]
TARGET = "salary"

X_train = train[FEATURE_COLS]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS]
y_test  = test[TARGET]
X_val   = val[FEATURE_COLS]
y_val   = val[TARGET]

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")
print(f"\nFeatures ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)

## 4. Train Champion Model (GradientBoostingRegressor)

In [ ]:
champion = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    random_state=42,
    verbose=0,
)
champion.fit(X_train, y_train)

val_pred  = champion.predict(X_val)
test_pred = champion.predict(X_test)

print(f"Val  R²  = {r2_score(y_val, val_pred):.4f}")
print(f"Val  MAE = ${mean_absolute_error(y_val, val_pred):,.0f}")
print(f"Test R²  = {r2_score(y_test, test_pred):.4f}")
print(f"Test MAE = ${mean_absolute_error(y_test, test_pred):,.0f}")

## 5. Global SHAP Values

In [ ]:
# Subsample test set for speed
shap_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
explainer   = shap.TreeExplainer(champion)
shap_values = explainer(shap_sample, check_additivity=False)

print(f"SHAP values computed for {len(shap_sample)} samples")

### 5a. SHAP Summary Plot (Beeswarm)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values.values,
    shap_sample,
    feature_names=FEATURE_COLS,
    show=False,
    plot_size=None,
)
plt.title("SHAP Summary Plot — Global Feature Impact on Salary", fontsize=14, pad=12)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "shap_summary_beeswarm.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shap_summary_beeswarm.png")

### 5b. SHAP Bar Plot (Mean |SHAP|)

In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
importance_df = pd.DataFrame(
    {"feature": FEATURE_COLS, "mean_abs_shap": mean_abs_shap}
).sort_values("mean_abs_shap", ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette("Blues_r", len(FEATURE_COLS))
ax.barh(
    importance_df["feature"][::-1],
    importance_df["mean_abs_shap"][::-1],
    color=colors[::-1],
)
ax.set_xlabel("Mean |SHAP value|  (impact on model output in USD)", fontsize=12)
ax.set_title("Global Feature Importance — Mean |SHAP| Values", fontsize=14)
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "shap_bar_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Top-5 features by mean |SHAP|:")
print(importance_df.head())

## 6. Local SHAP — Waterfall Plots (5 Representative Profiles)

In [ ]:
test_full = test.copy()
test_full["pred"] = champion.predict(X_test)

# Select 5 profiles
profiles_idx = []
profiles_idx.append(test_full["pred"].nsmallest(10).index[-1])                           # Low
profiles_idx.append((test_full["pred"] - test_full["pred"].median()).abs().idxmin())      # Median
profiles_idx.append(test_full["pred"].nlargest(10).index[-1])                            # High

dev_mask     = test_full["developing_country_flag"] > 0
dev_profiles = test_full[dev_mask].nlargest(2, "pred").index.tolist()
profiles_idx += dev_profiles[:2]   # 2 developing-country profiles

profile_labels = [
    "Low Salary Profile",
    "Median Salary Profile",
    "High Salary Profile",
    "Developing Country — Profile A",
    "Developing Country — Profile B",
]

local_shap_exp = explainer(X_test.loc[profiles_idx], check_additivity=False)
print(f"Selected {len(profiles_idx)} profiles for waterfall plots")

In [ ]:
for i, (idx, label) in enumerate(zip(profiles_idx, profile_labels)):
    fig, ax = plt.subplots(figsize=(11, 6))
    shap.waterfall_plot(local_shap_exp[i], max_display=15, show=False)
    plt.title(
        f"Local SHAP Waterfall — {label}\n"
        f"Predicted salary: ${test_full.loc[idx, 'pred']:,.0f}  |  "
        f"Actual: ${test_full.loc[idx, 'salary']:,.0f}",
        fontsize=12,
    )
    plt.tight_layout()
    fname = os.path.join(OUTPUT_DIR, f"shap_waterfall_profile_{i+1}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Profile {i+1}: {label} — saved")

## 7. Export SHAP Values to Parquet

In [ ]:
shap_df = pd.DataFrame(shap_values.values, columns=FEATURE_COLS, index=shap_sample.index)
shap_df["base_value"]       = shap_values.base_values
shap_df["predicted_salary"] = champion.predict(shap_sample)
shap_df["actual_salary"]    = y_test.loc[shap_sample.index]

shap_out_path = os.path.join(OUTPUT_DIR, "shap_values.parquet")
shap_df.to_parquet(shap_out_path, index=True)

print(f"Saved: {shap_out_path}")
print(f"Shape: {shap_df.shape}")
shap_df.head()

## 8. Counterfactual Generation with DiCE-ML
**Question: What needs to change for an India worker to earn X% more?**

In [ ]:
# Setup DiCE
train_dice           = X_train.copy()
train_dice["salary"] = y_train.values

dice_data  = dice_ml.Data(
    dataframe=train_dice,
    continuous_features=FEATURE_COLS,
    outcome_name="salary",
)
dice_model = dice_ml.Model(model=champion, backend="sklearn", model_type="regressor")
exp        = dice_ml.Dice(dice_data, dice_model, method="random")

print("DiCE explainer ready")

In [ ]:
# Select 2 lowest-salary developing-country (India) profiles
chosen_dev = test_full[dev_mask].nsmallest(20, "pred").index.tolist()
chosen_dev = [chosen_dev[0], chosen_dev[10]]

VARS_TO_VARY = ["job_title", "experience_years", "education_level",
                "skills_count", "certifications", "company_size", "remote_work"]
feat_bounds  = {f: (X_train[f].min(), X_train[f].max()) for f in FEATURE_COLS}


def manual_counterfactuals(query_row, model, bounds, current_pred):
    """4 interpretable scenario-based counterfactuals."""
    scenarios = [
        {
            "label": "Senior role\n(↑ job_title, ↑ experience)",
            "changes": {"job_title": bounds["job_title"][1],
                        "experience_years": bounds["experience_years"][1]},
        },
        {
            "label": "Education + Skills\n(↑ education, ↑ skills)",
            "changes": {"education_level": bounds["education_level"][1],
                        "skills_count": bounds["skills_count"][1]},
        },
        {
            "label": "Large company\n+ Remote (↑ co. size, remote)",
            "changes": {"company_size": bounds["company_size"][1],
                        "remote_work": bounds["remote_work"][1]},
        },
        {
            "label": "All levers\n(maximize all factors)",
            "changes": {f: bounds[f][1] for f in VARS_TO_VARY},
        },
    ]
    results = []
    for sc in scenarios:
        q = query_row.copy()
        for feat, val in sc["changes"].items():
            q[feat] = val
        pred = model.predict(q.to_frame().T)[0]
        results.append({
            "Scenario": sc["label"],
            "Salary":   pred,
            "Gain":     pred - current_pred,
            "Gain%":    (pred - current_pred) / current_pred * 100,
        })
    return pd.DataFrame(results)


print("Helper function defined")

### 8a. Counterfactual — India Profile A

In [ ]:
cf_results_summary = []

for i, idx in enumerate(chosen_dev):
    query        = X_test.loc[idx].copy()
    current_pred = champion.predict(query.to_frame().T)[0]

    print(f"\n{'='*55}")
    print(f"Developing Country Profile {chr(65+i)} (India)")
    print(f"{'='*55}")
    print(f"Current predicted salary: ${current_pred:,.0f}")

    # Try DiCE
    desired_low  = current_pred * 1.25
    desired_high = current_pred * 2.5
    try:
        cf    = exp.generate_counterfactuals(
            query_instances=query.to_frame().T,
            total_CFs=4,
            desired_range=[desired_low, desired_high],
            features_to_vary=VARS_TO_VARY,
            random_seed=42,
        )
        cf_df = cf.cf_examples_list[0].final_cfs_df
        if cf_df is not None and len(cf_df) > 0:
            cf_df["predicted_salary_CF"] = champion.predict(cf_df[FEATURE_COLS])
            print(f"DiCE generated {len(cf_df)} counterfactuals")
    except Exception as e:
        print(f"DiCE: {e} — using manual scenarios")

    # Manual scenario counterfactuals
    cf_manual = manual_counterfactuals(query, champion, feat_bounds, current_pred)
    display(cf_manual.style.format({"Salary": "${:,.0f}", "Gain": "${:,.0f}", "Gain%": "{:.1f}%"}))

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        f"Counterfactual Analysis — India Worker Profile {chr(65+i)}\n"
        f"'What changes are needed to earn more?'\n"
        f"Current Salary: ${current_pred:,.0f}",
        fontsize=13, y=1.01,
    )

    # Panel A
    ax       = axes[0]
    salaries = cf_manual["Salary"].values
    gains    = cf_manual["Gain"].values
    bar_cols = ["#2ecc71" if g > 0 else "#e74c3c" for g in gains]
    bars     = ax.barh(cf_manual["Scenario"], salaries, color=bar_cols, alpha=0.85, height=0.5)
    ax.axvline(current_pred, color="#e74c3c", linewidth=2, linestyle="--",
               label=f"Current: ${current_pred:,.0f}")
    ax.set_xlabel("Predicted Salary (USD)", fontsize=11)
    ax.set_title("Counterfactual Salaries vs. Current", fontsize=11)
    ax.legend(fontsize=9)
    for bar, gain_pct in zip(bars, cf_manual["Gain%"].values):
        ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
                f"+{gain_pct:.0f}%", va="center", fontsize=9, color="#1a5276")
    ax.set_xlim(0, max(salaries) * 1.15)

    # Panel B
    ax2          = axes[1]
    short_labels = ["Senior Role", "Edu+Skills", "Lg Co+Remote", "All Levers"]
    bar2         = ax2.bar(short_labels, gains, color=bar_cols, alpha=0.85)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_ylabel("Salary Gain (USD)", fontsize=11)
    ax2.set_title("Salary Gain per Scenario", fontsize=11)
    for j, (bar, g, sl) in enumerate(zip(bar2, gains, short_labels)):
        ax2.text(j, g + max(gains) * 0.02, f"${g:,.0f}", ha="center", fontsize=9, fontweight="bold")
        ax2.text(j, -max(gains) * 0.08, sl, ha="center", fontsize=8, color="grey")

    plt.tight_layout()
    fname = os.path.join(OUTPUT_DIR, f"counterfactual_profile_{chr(65+i)}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: counterfactual_profile_{chr(65+i)}.png")

    cf_results_summary.append({
        "profile":          f"India Worker Profile {chr(65+i)}",
        "current_salary":   current_pred,
        "best_cf_salary":   cf_manual["Salary"].max(),
        "best_salary_gain": cf_manual["Gain"].max(),
        "best_gain_pct":    cf_manual["Gain%"].max(),
    })

## 9. Summary

In [ ]:
summary_df = pd.DataFrame(cf_results_summary)
display(summary_df.style.format({
    "current_salary":   "${:,.0f}",
    "best_cf_salary":   "${:,.0f}",
    "best_salary_gain": "${:,.0f}",
    "best_gain_pct":    "{:.0f}%",
}))

In [ ]:
print("Output files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size  = os.path.getsize(fpath)
    print(f"  {f:50s}  {size/1024:8.1f} KB")

print("\n✓ Notebook 4 complete.")